# exp_20260514_028_parent_child_mlp_shared010_min

Source: `code_shared_010.txt`

Full Parent-Child MLP shared010 adaptation. The main blend artifact uses children-only predictions, matching the original shared010 main submission.

Note: 027 is already used by `exp_20260514_027_realmlp_shared009_min`, so shared010 is assigned experiment ID 028.


In [1]:
# ============================================================
# PS S6E5 - exp_20260514_028_parent_child_mlp_shared010_min
#
# Source:
#   code_shared_010.txt
#
# Purpose:
#   Full reproduction/adaptation of shared010 Parent-Child MLP ensemble.
#   This is treated as a separate NN family candidate from RealMLP/TabM/GBDT.
#
# Notes:
#   - Normalized_TyreLife is dropped if present.
#   - OOF/pred npy files and memo_draft.yml are added for blend workflow.
#   - Main blend artifact uses children-only predictions, matching the original shared010 main submission.
# ============================================================

In [2]:
import sys
import subprocess

try:
    import rtdl_num_embeddings  # noqa: F401
except Exception:
    print("Installing rtdl-num-embeddings...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rtdl-num-embeddings"])

# ============================================================
# 5-FOLD CLEAN PARENT + CHILDREN ENSEMBLE
# Binary classification: PitNextLap
#
# Each fold:
#   - train parent on fold_train + orig
#   - validate on fold_val
#   - create 5 children from parent
#   - each child inherits random hidden neurons from parent
#   - remaining hidden neurons are reset and retrained
#   - progressive freezing is used for both parent and children
#
# Final:
#   - submission.csv = average of all fold children ensembles
#
# Writes:
#   submission.csv
#   submission_children_only.csv
#   submission_all_models_mean.csv
#   submission_parent_only.csv
# ============================================================

import copy
import math
import random
import gc
from dataclasses import dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import QuantileTransformer, KBinsDiscretizer
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import rtdl_num_embeddings
    HAS_RTDL_NUM_EMB = True
except Exception as e:
    HAS_RTDL_NUM_EMB = False
    print("Could not import rtdl_num_embeddings. Falling back to raw numerical inputs.")
    print("Import error:", repr(e))

Installing rtdl-num-embeddings...


In [3]:
# ============================================================
# 0) CONFIG
# ============================================================

SEED = 21

TRAIN_PATHS = [
    "/kaggle/input/competitions/playground-series-s6e5/train.csv",
    "/kaggle/input/playground-series-s6e5/train.csv",
]

TEST_PATHS = [
    "/kaggle/input/competitions/playground-series-s6e5/test.csv",
    "/kaggle/input/playground-series-s6e5/test.csv",
]

ORIG_PATH = "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv"

ID_COL = "id"
TARGET = "PitNextLap"

EXP_ID = "exp_20260514_028_parent_child_mlp_shared010_min"
OUTDIR = Path(f"/kaggle/working/{EXP_ID}")
OUTDIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 5
N_CHILDREN_PER_FOLD = 10

N_HIDDEN_LAYERS = 3
PARENT_HIDDEN = 300
PARENT_DROPOUT = 0.10
BATCH_SIZE = 4096

PARENT_FREEZE_STEP_FRAC = 0.25
PARENT_MIN_TRAINABLE = 40
PARENT_LR = 8e-4
PARENT_WEIGHT_DECAY = 1e-5
PARENT_MAX_EPOCHS_PER_STAGE = 4
PARENT_PATIENCE_PER_STAGE = 2

CHILD_INHERIT_FRAC_LOW = 0.20
CHILD_INHERIT_FRAC_HIGH = 0.45
CHILD_MAX_EPOCHS_PER_STAGE = 4
CHILD_PATIENCE_PER_STAGE = 2
CHILD_WEIGHT_DECAY = 1e-5
CHILD_FREEZE_STEP_FRAC = 0.30

USE_NUM_EMBEDDINGS = HAS_RTDL_NUM_EMB

NUM_EMB_SPECS = [
    {"n_bins": 32, "d_emb": 8},
    {"n_bins": 128, "d_emb": 8},
    {"n_bins": 128, "d_emb": 16},
    {"n_bins": 256, "d_emb": 32},
    {"n_bins": 256, "d_emb": 64},
]

OUTPUT_FILE = str(OUTDIR / f"submission_{EXP_ID}.csv")

In [4]:
# ============================================================
# 1) SEED / DEVICE
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def auc_score(y_true_np, y_prob_np):
    y_true_np = np.asarray(y_true_np).astype(int)
    y_prob_np = np.asarray(y_prob_np, dtype=np.float64)
    return float(roc_auc_score(y_true_np, y_prob_np))


def first_existing_path(paths):
    for p in paths:
        if Path(p).exists():
            return p
    raise FileNotFoundError(f"No path exists from: {paths}")

Device: cuda
GPU: Tesla T4


In [5]:
# ============================================================
# 2) LOAD DATA
# ============================================================

TRAIN_PATH = first_existing_path(TRAIN_PATHS)
TEST_PATH = first_existing_path(TEST_PATHS)

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print("competition train:", train_raw.shape)
print("competition test :", test_raw.shape)

if Path(ORIG_PATH).exists():
    orig_raw = pd.read_csv(ORIG_PATH)
    print("external orig:", orig_raw.shape)
else:
    orig_raw = None
    print("No external orig found.")

for name, df in [("train_raw", train_raw), ("test_raw", test_raw)]:
    if "Normalized_TyreLife" in df.columns:
        df.drop(columns=["Normalized_TyreLife"], inplace=True)
        print(f"Dropped Normalized_TyreLife from {name}")

if orig_raw is not None and "Normalized_TyreLife" in orig_raw.columns:
    orig_raw.drop(columns=["Normalized_TyreLife"], inplace=True)
    print("Dropped Normalized_TyreLife from orig_raw")

X_all_raw = train_raw.drop([ID_COL, TARGET], axis=1).copy()
y_all = train_raw[TARGET].astype(int).copy()
train_ids_all = train_raw[ID_COL].copy()

X_test_raw = test_raw.drop([ID_COL], axis=1).copy()
test_id = test_raw[ID_COL].copy()

if orig_raw is not None:
    y_orig = orig_raw[TARGET].astype(int).copy()
    X_orig_raw = orig_raw.drop([TARGET], axis=1).copy()
    X_orig_raw = X_orig_raw.reindex(columns=X_all_raw.columns)
else:
    y_orig = None
    X_orig_raw = None

base_cat_cols = X_all_raw.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
base_num_cols = [c for c in X_all_raw.columns if c not in base_cat_cols]

print("base_cat_cols:", base_cat_cols)
print("base_num_cols:", base_num_cols)

competition train: (439140, 16)
competition test : (188165, 15)
external orig: (101371, 16)
Dropped Normalized_TyreLife from orig_raw
base_cat_cols: ['Driver', 'Compound', 'Race']
base_num_cols: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']


In [6]:
# ============================================================
# 3) FEATURE ENGINEERING
# ============================================================

important_combos = [
    ("Race", "Compound"),
    ("Race", "Year"),
]


def make_feature_engineer():
    category_map = {}

    def feature_engineering(df, fit=False):
        df = df.copy()

        if "LapNumber" in df.columns and "RaceProgress" in df.columns:
            df["_LapNumber_/_RaceProgress"] = (
                df["LapNumber"] / (df["RaceProgress"] + 1e-6)
            ).astype("float32")

        if "TyreLife" in df.columns and "LapNumber" in df.columns:
            df["_TyreLife_/_LapNumber"] = (
                df["TyreLife"] / df["LapNumber"].clip(lower=1)
            ).astype("float32")

        floor_cat_source_cols = base_num_cols.copy()

        for extra_col in ["_LapNumber_/_RaceProgress", "_TyreLife_/_LapNumber"]:
            if extra_col in df.columns:
                floor_cat_source_cols.append(extra_col)

        for col in floor_cat_source_cols:
            if col not in df.columns:
                continue

            cat_name = f"{col}_cat_" if col in base_num_cols else f"{col[1:]}_cat_"
            floored = np.floor(df[col].astype(float))

            if fit:
                codes, uniques = pd.factorize(floored, sort=False)
                category_map[col] = uniques
            else:
                uniques = category_map[col]
                code_map = {cat: i for i, cat in enumerate(uniques)}
                codes = pd.Series(floored, index=df.index).map(code_map).fillna(-1).astype("int32")

            df[cat_name] = codes.astype("int32").astype(str)

        count_source_cols = base_cat_cols.copy()

        for maybe_col in ["Year_cat_", "PitStop_cat_"]:
            if maybe_col in df.columns and maybe_col not in count_source_cols:
                count_source_cols.append(maybe_col)

        for col in count_source_cols:
            if col not in df.columns:
                continue

            count_name = f"_{col}_count" if col in base_cat_cols else f"_{col[:-1]}_count"

            if fit:
                count_map = df[col].value_counts(dropna=False)
                category_map[count_name] = count_map
            else:
                count_map = category_map[count_name]

            df[count_name] = df[col].map(count_map).fillna(0).astype("int32")

        bin_config = {
            "RaceProgress": [200],
            "LapTime (s)": [7],
        }

        for col, bins_list in bin_config.items():
            if col not in df.columns:
                continue

            for n_bins in bins_list:
                bin_name = f"{col}_{n_bins}_quantile_bin_"

                if fit:
                    kb = KBinsDiscretizer(
                        n_bins=n_bins,
                        encode="ordinal",
                        strategy="quantile",
                        subsample=None,
                    )
                    binned = kb.fit_transform(df[[col]]).ravel().astype("int32")
                    category_map[bin_name] = kb
                else:
                    kb = category_map[bin_name]
                    binned = kb.transform(df[[col]]).ravel().astype("int32")

                df[bin_name] = binned.astype("int32").astype(str)

        combo_names = []

        for cols in important_combos:
            if not all(c in df.columns for c in cols):
                continue

            combo_name = "_".join(cols) + "_"
            combo_names.append(combo_name)

            combo_series = df[cols[0]].astype(str)

            for col in cols[1:]:
                combo_series = combo_series + "_" + df[col].astype(str)

            if fit:
                codes, uniques = pd.factorize(combo_series, sort=False)
                category_map[combo_name] = uniques
            else:
                uniques = category_map[combo_name]
                code_map = {cat: i for i, cat in enumerate(uniques)}
                codes = combo_series.map(code_map).fillna(-1).astype("int32")

            df[combo_name] = codes.astype("int32").astype(str)

        new_cat_cols = [col for col in df.columns if col.endswith("_")]
        new_num_cols = [col for col in df.columns if col.startswith("_")]

        return df, new_cat_cols, new_num_cols, combo_names

    return feature_engineering

In [7]:
# ============================================================
# 4) MODEL UTILITIES
# ============================================================

@dataclass
class FreezePlan:
    frozen_idx: np.ndarray
    trainable_idx: np.ndarray


class ProgressiveFreezePlanner1D:
    def __init__(self, width, freeze_step_frac, min_trainable, seed=0):
        self.width = int(width)
        self.freeze_step_frac = float(freeze_step_frac)
        self.min_trainable = int(min_trainable)
        self.rng = np.random.default_rng(seed)
        self.plans = self._make_plans()

    def _make_plans(self):
        plans = []
        frozen = np.array([], dtype=np.int64)

        while True:
            all_idx = np.arange(self.width, dtype=np.int64)
            trainable = np.setdiff1d(all_idx, frozen, assume_unique=False)

            if len(trainable) <= self.min_trainable:
                break

            plans.append(FreezePlan(frozen.copy(), trainable.copy()))

            k = max(1, int(round(self.freeze_step_frac * len(trainable))))
            k = min(k, len(trainable))

            new_chunk = np.sort(
                self.rng.choice(trainable, size=k, replace=False).astype(np.int64)
            )

            frozen = np.sort(np.concatenate([frozen, new_chunk]).astype(np.int64))

        if len(plans) == 0:
            all_idx = np.arange(self.width, dtype=np.int64)
            plans.append(FreezePlan(np.array([], dtype=np.int64), all_idx))

        return plans


def default_emb_dim(cardinality):
    return int(min(50, max(4, round((cardinality ** 0.5) * 2))))


class ParentTabMLP(nn.Module):
    def __init__(
        self,
        cat_sizes,
        d_num,
        hidden,
        n_hidden_layers,
        dropout,
        use_num_embeddings,
        num_emb_dim_flat,
        make_num_emb_module,
        cats,
    ):
        super().__init__()

        self.d_num = int(d_num)
        self.hidden = int(hidden)
        self.n_hidden_layers = int(n_hidden_layers)
        self.use_num_embeddings = bool(use_num_embeddings)
        self.cats = list(cats)

        if self.n_hidden_layers != 3:
            raise ValueError("This script expects exactly 3 hidden layers.")

        self.num_embs = make_num_emb_module()

        self.emb_layers = nn.ModuleDict()
        emb_out_dim = 0

        for c in self.cats:
            n = int(cat_sizes[c])
            d = default_emb_dim(n)
            self.emb_layers[c] = nn.Embedding(n, d)
            emb_out_dim += d

        num_dim = int(num_emb_dim_flat) if self.use_num_embeddings else self.d_num
        in_dim = num_dim + emb_out_dim

        if in_dim == 0:
            raise ValueError("Need at least one numerical or categorical feature.")

        self.in_dim = int(in_dim)
        self.ln_in = nn.LayerNorm(in_dim)

        self.hidden_linears = nn.ModuleList()
        self.hidden_lns = nn.ModuleList()

        dims = [in_dim, hidden, hidden]

        for in_d in dims:
            self.hidden_linears.append(nn.Linear(in_d, hidden))
            self.hidden_lns.append(nn.LayerNorm(hidden))

        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.out = nn.Linear(hidden, 1)

    def build_input(self, x_num, x_cat):
        chunks = []

        if self.d_num > 0:
            if self.use_num_embeddings:
                emb_chunks = []

                for emb in self.num_embs:
                    emb_chunks.append(emb(x_num).flatten(1))

                chunks.append(torch.cat(emb_chunks, dim=1))
            else:
                chunks.append(x_num)

        if len(self.cats) > 0:
            cat_embs = []

            for j, c in enumerate(self.cats):
                cat_embs.append(self.emb_layers[c](x_cat[:, j]))

            chunks.extend(cat_embs)

        x = torch.cat(chunks, dim=1)
        return self.ln_in(x)

    def forward(self, x_num, x_cat):
        h = self.build_input(x_num, x_cat)

        for i in range(self.n_hidden_layers):
            h = self.hidden_linears[i](h)
            h = self.hidden_lns[i](h)
            h = F.relu(h)
            h = self.drop(h)

        return self.out(h).squeeze(1)


def zero_grads_for_frozen_rows(model, frozen_idx):
    if len(frozen_idx) == 0:
        return

    frozen_t = torch.tensor(frozen_idx, dtype=torch.long, device=device)

    for layer_idx in range(model.n_hidden_layers):
        lin = model.hidden_linears[layer_idx]
        ln = model.hidden_lns[layer_idx]

        if lin.weight.grad is not None:
            lin.weight.grad[frozen_t, :] = 0.0

            if layer_idx > 0:
                lin.weight.grad[:, frozen_t] = 0.0

        if lin.bias.grad is not None:
            lin.bias.grad[frozen_t] = 0.0

        if ln.weight.grad is not None:
            ln.weight.grad[frozen_t] = 0.0

        if ln.bias.grad is not None:
            ln.bias.grad[frozen_t] = 0.0

    if model.out.weight.grad is not None:
        model.out.weight.grad[:, frozen_t] = 0.0


@torch.no_grad()
def predict_logits(model, Xn, Xc, bs=8192):
    model.eval()
    out = []

    for i in range(0, Xn.shape[0], bs):
        sl = slice(i, min(i + bs, Xn.shape[0]))
        out.append(model(Xn[sl], Xc[sl]))

    return torch.cat(out, dim=0)


@torch.no_grad()
def predict_proba_model(model, Xn, Xc, bs=8192):
    return torch.sigmoid(predict_logits(model, Xn, Xc, bs=bs))


@torch.no_grad()
def predict_ensemble_mean_proba(models, Xn, Xc, bs=8192):
    probs = [predict_proba_model(m, Xn, Xc, bs=bs) for m in models]
    return torch.stack(probs, dim=1).mean(dim=1)


def train_parent_progressive_freeze(
    model,
    Xn_train,
    Xc_train,
    y_train,
    Xn_val,
    Xc_val,
    y_val_np,
    *,
    freeze_plans,
    lr,
    weight_decay,
    batch_size,
    max_epochs_per_stage,
    patience_per_stage,
    tag,
):
    global_best_auc = -float("inf")
    global_best_state = copy.deepcopy(model.state_dict())

    for stage_idx, stage_plan in enumerate(freeze_plans, start=1):
        print("\n------------------------------------------------")
        print(f"{tag} STAGE {stage_idx}/{len(freeze_plans)}")
        print("------------------------------------------------")

        frozen_idx = stage_plan.frozen_idx

        print(f"frozen={len(frozen_idx)} | trainable={len(stage_plan.trainable_idx)}")

        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

        best_stage_auc = -float("inf")
        best_stage_state = copy.deepcopy(model.state_dict())
        best_stage_epoch = 0
        no_imp = 0
        N = Xn_train.shape[0]

        for ep in range(1, max_epochs_per_stage + 1):
            model.train()
            perm = torch.randperm(N, device=device)

            loss_sum = 0.0
            nb = 0

            for start in range(0, N, batch_size):
                b = perm[start:start + batch_size]

                opt.zero_grad(set_to_none=True)
                logits = model(Xn_train[b], Xc_train[b])
                loss = F.binary_cross_entropy_with_logits(logits, y_train[b])
                loss.backward()
                zero_grads_for_frozen_rows(model, frozen_idx)
                opt.step()

                loss_sum += float(loss.item())
                nb += 1

            val_prob = predict_proba_model(model, Xn_val, Xc_val).detach().cpu().numpy()
            val_auc = auc_score(y_val_np, val_prob)

            if val_auc > best_stage_auc + 1e-12:
                best_stage_auc = val_auc
                best_stage_state = copy.deepcopy(model.state_dict())
                best_stage_epoch = ep
                no_imp = 0
            else:
                no_imp += 1

            print(
                f"[{tag} Stage {stage_idx}] Epoch {ep:02d}/{max_epochs_per_stage} | "
                f"train_bce={loss_sum / max(1, nb):.6f} | "
                f"val_AUC={val_auc:.6f} "
                f"(best {best_stage_auc:.6f}@{best_stage_epoch})"
            )

            if no_imp >= patience_per_stage:
                print(f"[{tag} Stage {stage_idx}] Early stop at epoch {ep}.")
                break

        model.load_state_dict(best_stage_state)

        if best_stage_auc > global_best_auc + 1e-12:
            global_best_auc = best_stage_auc
            global_best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(global_best_state)

    return model, global_best_auc


@torch.no_grad()
def _kaiming_reset_selected_rows(weight, row_idx):
    if row_idx.numel() == 0:
        return

    tmp = weight[row_idx, :].clone()
    nn.init.kaiming_uniform_(tmp, a=math.sqrt(5))
    weight[row_idx, :] = tmp


@torch.no_grad()
def _kaiming_reset_selected_cols(weight, col_idx):
    if col_idx.numel() == 0:
        return

    tmp = weight[:, col_idx].clone()
    nn.init.kaiming_uniform_(tmp, a=math.sqrt(5))
    weight[:, col_idx] = tmp


@torch.no_grad()
def build_child_from_parent(final_parent, inherit_frac, seed):
    rng = np.random.default_rng(seed)
    child = copy.deepcopy(final_parent).to(device)

    for p in child.parameters():
        p.requires_grad = True

    hidden = child.hidden
    all_idx = np.arange(hidden, dtype=np.int64)

    n_keep = max(1, min(hidden - 1, int(round(inherit_frac * hidden))))

    inherited_idx = np.sort(
        rng.choice(all_idx, size=n_keep, replace=False).astype(np.int64)
    )

    reset_idx = np.setdiff1d(all_idx, inherited_idx, assume_unique=False)

    reset_t = torch.tensor(reset_idx, dtype=torch.long, device=device)

    if reset_t.numel() > 0:
        for layer_idx in range(child.n_hidden_layers):
            lin = child.hidden_linears[layer_idx]
            ln = child.hidden_lns[layer_idx]

            _kaiming_reset_selected_rows(lin.weight, reset_t)
            lin.bias[reset_t] = 0.0

            ln.weight[reset_t] = 1.0
            ln.bias[reset_t] = 0.0

            if layer_idx > 0:
                _kaiming_reset_selected_cols(lin.weight, reset_t)

        _kaiming_reset_selected_cols(child.out.weight, reset_t)

    return child, inherited_idx, reset_idx


def train_child_progressive_freeze(
    child,
    inherited_idx,
    Xn_train,
    Xc_train,
    y_train,
    Xn_val,
    Xc_val,
    y_val_eval_np,
    parent_val_pred_np,
    *,
    child_plans,
    lr,
    weight_decay,
    batch_size,
    max_epochs_per_stage,
    patience_per_stage,
    tag,
    ensemble_parent_weight=0.5,
):
    for p in child.parameters():
        p.requires_grad = True

    inherited_set = set(int(x) for x in inherited_idx.tolist())

    best_global_auc = -float("inf")
    best_global_state = copy.deepcopy(child.state_dict())

    for stage_idx, stage_plan in enumerate(child_plans, start=1):
        frozen_reset_idx = stage_plan.frozen_idx

        frozen_total = np.array(
            sorted(list(inherited_set.union(set(int(x) for x in frozen_reset_idx.tolist())))),
            dtype=np.int64,
        )

        print("\n------------------------------------------------")
        print(f"{tag} STAGE {stage_idx}/{len(child_plans)}")
        print("------------------------------------------------")
        print(
            f"inherited_frozen={len(inherited_idx)} | "
            f"reset_frozen={len(frozen_reset_idx)} | "
            f"reset_trainable={len(stage_plan.trainable_idx)}"
        )

        opt = torch.optim.AdamW(child.parameters(), lr=lr, weight_decay=weight_decay)

        best_stage_auc = -float("inf")
        best_stage_state = copy.deepcopy(child.state_dict())
        best_stage_epoch = 0
        no_imp = 0
        N = Xn_train.shape[0]

        for ep in range(1, max_epochs_per_stage + 1):
            child.train()
            perm = torch.randperm(N, device=device)

            loss_sum = 0.0
            nb = 0

            for start in range(0, N, batch_size):
                b = perm[start:start + batch_size]

                opt.zero_grad(set_to_none=True)
                logits = child(Xn_train[b], Xc_train[b])
                loss = F.binary_cross_entropy_with_logits(logits, y_train[b])
                loss.backward()
                zero_grads_for_frozen_rows(child, frozen_total)
                opt.step()

                loss_sum += float(loss.item())
                nb += 1

            child_val_prob = predict_proba_model(child, Xn_val, Xc_val).detach().cpu().numpy()

            ens_val_prob = (
                ensemble_parent_weight * parent_val_pred_np
                + (1.0 - ensemble_parent_weight) * child_val_prob
            )

            val_auc = auc_score(y_val_eval_np, ens_val_prob)

            if val_auc > best_stage_auc + 1e-12:
                best_stage_auc = val_auc
                best_stage_state = copy.deepcopy(child.state_dict())
                best_stage_epoch = ep
                no_imp = 0
            else:
                no_imp += 1

            print(
                f"[{tag} Stage {stage_idx}] Epoch {ep:02d}/{max_epochs_per_stage} | "
                f"train_bce={loss_sum / max(1, nb):.6f} | "
                f"parent_child_val_AUC={val_auc:.6f} "
                f"(best {best_stage_auc:.6f}@{best_stage_epoch})"
            )

            if no_imp >= patience_per_stage:
                print(f"[{tag} Stage {stage_idx}] Early stop at epoch {ep}.")
                break

        child.load_state_dict(best_stage_state)

        if best_stage_auc > best_global_auc + 1e-12:
            best_global_auc = best_stage_auc
            best_global_state = copy.deepcopy(child.state_dict())

    child.load_state_dict(best_global_state)

    return child, best_global_auc

In [8]:
# ============================================================
# 5) FOLD TRAINING FUNCTION
# ============================================================

def run_one_fold(fold, tr_idx, val_idx):
    print("\n\n========================================================")
    print(f"====================== FOLD {fold}/{N_FOLDS} ======================")
    print("========================================================")

    fold_seed = SEED + fold * 1000

    feature_engineering = make_feature_engineer()

    df_tr_raw = X_all_raw.iloc[tr_idx].copy()
    df_val_raw = X_all_raw.iloc[val_idx].copy()

    y_tr_raw = y_all.iloc[tr_idx].reset_index(drop=True).copy()
    y_val_raw = y_all.iloc[val_idx].reset_index(drop=True).copy()

    train_ids_fold = train_ids_all.iloc[tr_idx].reset_index(drop=True).copy()
    val_ids_fold = train_ids_all.iloc[val_idx].reset_index(drop=True).copy()

    X_tr_fe, new_cat_cols, new_num_cols, combo_names = feature_engineering(df_tr_raw, fit=True)
    X_val_fe, _, _, _ = feature_engineering(df_val_raw, fit=False)
    X_test_fe, _, _, _ = feature_engineering(X_test_raw, fit=False)

    cat_cols = base_cat_cols.copy()

    if "PitStop" in X_tr_fe.columns and "PitStop" not in cat_cols:
        cat_cols.append("PitStop")

    cat_cols += new_cat_cols

    num_cols = base_num_cols.copy()
    num_cols += new_num_cols

    cat_cols = list(dict.fromkeys(cat_cols))
    num_cols = list(dict.fromkeys(num_cols))

    cat_cols = [c for c in cat_cols if c in X_tr_fe.columns]
    num_cols = [c for c in num_cols if c in X_tr_fe.columns]

    df_train_base = X_tr_fe.copy()
    df_train_base[ID_COL] = train_ids_fold.values
    df_train_base[TARGET] = y_tr_raw.values

    df_val = X_val_fe.copy()
    df_val[ID_COL] = val_ids_fold.values
    df_val[TARGET] = y_val_raw.values

    df_test = X_test_fe.copy()
    df_test[ID_COL] = test_id.values

    if X_orig_raw is not None:
        X_orig_fe, _, _, _ = feature_engineering(X_orig_raw, fit=False)

        df_orig = X_orig_fe.copy()
        df_orig[ID_COL] = np.arange(-len(df_orig), 0)
        df_orig[TARGET] = y_orig.values
        df_orig = df_orig.reindex(columns=df_train_base.columns)

        df_train = pd.concat([df_train_base, df_orig], axis=0).reset_index(drop=True)
    else:
        df_train = df_train_base.reset_index(drop=True).copy()

    df_val = df_val.reset_index(drop=True).copy()
    df_test = df_test.reset_index(drop=True).copy()

    print("fold train base:", df_train_base.shape)
    print("fold train final:", df_train.shape)
    print("fold val        :", df_val.shape)
    print("fold test       :", df_test.shape)

    print("cat_cols:", len(cat_cols))
    print("num_cols:", len(num_cols))
    print("combo_names:", combo_names)

    numericals = num_cols
    CATS = cat_cols

    # --------------------------
    # Numerical preprocessing
    # --------------------------

    if len(numericals) > 0:
        Xn_train_raw = df_train[numericals].astype(np.float32).values

        noise = np.random.default_rng(fold_seed).normal(
            0.0,
            1e-5,
            Xn_train_raw.shape,
        ).astype(np.float32)

        qt = QuantileTransformer(
            n_quantiles=max(min(len(df_train) // 30, 1000), 10),
            output_distribution="normal",
            subsample=10**9,
            random_state=fold_seed,
        )

        qt.fit(Xn_train_raw + noise)

        for df_ in [df_train, df_val, df_test]:
            df_[numericals] = qt.transform(
                df_[numericals].astype(np.float32).values
            ).astype(np.float32)

    # --------------------------
    # Categorical encoding
    # --------------------------

    cat_maps, cat_sizes = {}, {}

    for c in CATS:
        tr_vals = df_train[c].astype("string").fillna("__NA__")
        uniques = tr_vals.unique().tolist()
        mapping = {v: i + 1 for i, v in enumerate(uniques)}
        cat_maps[c] = mapping
        cat_sizes[c] = len(mapping) + 1

    def encode_cats(df):
        if len(CATS) == 0:
            return np.zeros((len(df), 0), dtype=np.int64)

        cols = []

        for c in CATS:
            vals = df[c].astype("string").fillna("__NA__")
            m = cat_maps[c]
            enc = vals.map(lambda v: m.get(v, 0)).astype(np.int64).values.reshape(-1, 1)
            cols.append(enc)

        return np.concatenate(cols, axis=1)

    def df_to_arrays(df, has_target=True):
        if len(numericals) > 0:
            X_num = df[numericals].astype(np.float32).values
        else:
            X_num = np.zeros((len(df), 0), dtype=np.float32)

        X_cat = encode_cats(df)

        if has_target:
            y = df[TARGET].astype(np.float32).values
            return X_num, X_cat, y

        return X_num, X_cat

    def arrays_to_tensors(X_num, X_cat, y=None):
        Xn = torch.tensor(X_num, dtype=torch.float32, device=device)
        Xc = torch.tensor(X_cat, dtype=torch.long, device=device)

        if y is None:
            return Xn, Xc

        yt = torch.tensor(y, dtype=torch.float32, device=device)
        return Xn, Xc, yt

    X_train_num, X_train_cat, y_train_np = df_to_arrays(df_train, has_target=True)
    X_val_num, X_val_cat, y_val_np_float = df_to_arrays(df_val, has_target=True)
    X_test_num, X_test_cat = df_to_arrays(df_test, has_target=False)

    Xn_train, Xc_train, y_train_t = arrays_to_tensors(X_train_num, X_train_cat, y_train_np)
    Xn_val, Xc_val, y_val_t = arrays_to_tensors(X_val_num, X_val_cat, y_val_np_float)
    Xn_test, Xc_test = arrays_to_tensors(X_test_num, X_test_cat)

    Dnum = Xn_train.shape[1]
    y_val_np = y_val_t.detach().cpu().numpy()

    print("Dnum =", Dnum, "| #cats =", len(CATS))

    # --------------------------
    # Num embeddings per fold
    # --------------------------

    use_num_embeddings_fold = bool(USE_NUM_EMBEDDINGS and Dnum > 0)

    if use_num_embeddings_fold:
        bins_list = []
        Xn_train_cpu = Xn_train.detach().cpu()

        for spec in NUM_EMB_SPECS:
            b = rtdl_num_embeddings.compute_bins(
                Xn_train_cpu,
                n_bins=int(spec["n_bins"]),
            )
            bins_list.append([t.to(device) for t in b])

        num_emb_dim_flat = Dnum * sum(int(s["d_emb"]) for s in NUM_EMB_SPECS)

        def make_multi_num_emb_module():
            mods = nn.ModuleList()

            for spec, bins in zip(NUM_EMB_SPECS, bins_list):
                mods.append(
                    rtdl_num_embeddings.PiecewiseLinearEmbeddings(
                        bins=bins,
                        d_embedding=int(spec["d_emb"]),
                        activation=False,
                        version="B",
                    )
                )

            return mods.to(device)

    else:
        use_num_embeddings_fold = False
        num_emb_dim_flat = Dnum

        def make_multi_num_emb_module():
            return nn.ModuleList().to(device)

    print("USE_NUM_EMBEDDINGS fold:", use_num_embeddings_fold)
    print("NUM_EMB_DIM_FLAT:", num_emb_dim_flat)

    # --------------------------
    # Parent
    # --------------------------

    print("\n\n======================================")
    print(f"===== FOLD {fold} TRAIN PARENT =======")
    print("======================================")

    parent_model = ParentTabMLP(
        cat_sizes=cat_sizes,
        d_num=Dnum,
        hidden=PARENT_HIDDEN,
        n_hidden_layers=N_HIDDEN_LAYERS,
        dropout=PARENT_DROPOUT,
        use_num_embeddings=use_num_embeddings_fold,
        num_emb_dim_flat=num_emb_dim_flat,
        make_num_emb_module=make_multi_num_emb_module,
        cats=CATS,
    ).to(device)

    parent_planner = ProgressiveFreezePlanner1D(
        width=PARENT_HIDDEN,
        freeze_step_frac=PARENT_FREEZE_STEP_FRAC,
        min_trainable=PARENT_MIN_TRAINABLE,
        seed=fold_seed + 111,
    )

    print(f"Parent stages = {len(parent_planner.plans)}")

    parent_model, parent_best_val_auc = train_parent_progressive_freeze(
        parent_model,
        Xn_train,
        Xc_train,
        y_train_t,
        Xn_val,
        Xc_val,
        y_val_np,
        freeze_plans=parent_planner.plans,
        lr=PARENT_LR,
        weight_decay=PARENT_WEIGHT_DECAY,
        batch_size=BATCH_SIZE,
        max_epochs_per_stage=PARENT_MAX_EPOCHS_PER_STAGE,
        patience_per_stage=PARENT_PATIENCE_PER_STAGE,
        tag=f"FOLD{fold}_PARENT",
    )

    with torch.no_grad():
        parent_val_pred = predict_proba_model(parent_model, Xn_val, Xc_val).detach().cpu().numpy()
        parent_test_pred = predict_proba_model(parent_model, Xn_test, Xc_test).detach().cpu().numpy()

    parent_val_auc = auc_score(y_val_np, parent_val_pred)

    print(f"[Fold {fold} Parent] VAL AUC = {parent_val_auc:.6f}")

    # --------------------------
    # Children
    # --------------------------

    print("\n\n======================================")
    print(f"===== FOLD {fold} TRAIN CHILDREN =====")
    print("======================================")

    rng_hp = np.random.default_rng(fold_seed + 999)

    child_lrs = rng_hp.uniform(5.5e-4, 8.0e-4, N_CHILDREN_PER_FOLD).tolist()
    child_dropouts = rng_hp.uniform(0.00, 0.06, N_CHILDREN_PER_FOLD).tolist()

    child_inherit_fracs = rng_hp.uniform(
        CHILD_INHERIT_FRAC_LOW,
        CHILD_INHERIT_FRAC_HIGH,
        N_CHILDREN_PER_FOLD,
    ).tolist()

    children = []
    child_val_scores = []

    for child_idx in range(N_CHILDREN_PER_FOLD):
        child_name = f"fold{fold}_child_{child_idx + 1}"

        print("\n=================================================")
        print(f"===== {child_name.upper()} =====")
        print("=================================================")

        child_model, inherited_idx, reset_idx = build_child_from_parent(
            final_parent=parent_model,
            inherit_frac=float(child_inherit_fracs[child_idx]),
            seed=fold_seed + 50000 + child_idx,
        )

        child_model.drop = (
            nn.Dropout(float(child_dropouts[child_idx]))
            if float(child_dropouts[child_idx]) > 0
            else nn.Identity()
        )

        child_model = child_model.to(device)

        for p in child_model.parameters():
            p.requires_grad = True

        child_planner_raw = ProgressiveFreezePlanner1D(
            width=len(reset_idx),
            freeze_step_frac=CHILD_FREEZE_STEP_FRAC,
            min_trainable=max(12, int(round(0.10 * len(reset_idx)))) if len(reset_idx) > 0 else 1,
            seed=fold_seed + 60000 + child_idx,
        )

        reset_idx_arr = np.asarray(reset_idx, dtype=np.int64)
        child_plans = []

        for plan in child_planner_raw.plans:
            child_plans.append(
                FreezePlan(
                    frozen_idx=reset_idx_arr[plan.frozen_idx]
                    if len(plan.frozen_idx) > 0
                    else np.array([], dtype=np.int64),
                    trainable_idx=reset_idx_arr[plan.trainable_idx]
                    if len(plan.trainable_idx) > 0
                    else np.array([], dtype=np.int64),
                )
            )

        print(
            f"inherit={len(inherited_idx)}/{PARENT_HIDDEN} | "
            f"reset={len(reset_idx)}/{PARENT_HIDDEN} | "
            f"stages={len(child_plans)} | "
            f"lr={child_lrs[child_idx]:.6f} | "
            f"dropout={child_dropouts[child_idx]:.3f}"
        )

        child_model, child_best_val_auc = train_child_progressive_freeze(
            child=child_model,
            inherited_idx=inherited_idx,
            Xn_train=Xn_train,
            Xc_train=Xc_train,
            y_train=y_train_t,
            Xn_val=Xn_val,
            Xc_val=Xc_val,
            y_val_eval_np=y_val_np,
            parent_val_pred_np=parent_val_pred,
            child_plans=child_plans,
            lr=float(child_lrs[child_idx]),
            weight_decay=CHILD_WEIGHT_DECAY,
            batch_size=BATCH_SIZE,
            max_epochs_per_stage=CHILD_MAX_EPOCHS_PER_STAGE,
            patience_per_stage=CHILD_PATIENCE_PER_STAGE,
            tag=child_name,
            ensemble_parent_weight=0.5,
        )

        with torch.no_grad():
            child_val_pred = predict_proba_model(child_model, Xn_val, Xc_val).detach().cpu().numpy()

        child_val_auc = auc_score(y_val_np, child_val_pred)
        child_ens_val_auc = auc_score(y_val_np, 0.5 * parent_val_pred + 0.5 * child_val_pred)

        print(
            f"{child_name} | "
            f"solo val={child_val_auc:.6f} | "
            f"parent+child val={child_ens_val_auc:.6f}"
        )

        children.append(child_model)

        child_val_scores.append({
            "child": child_idx + 1,
            "solo_val_auc": float(child_val_auc),
            "parent_child_val_auc": float(child_ens_val_auc),
            "inherit": int(len(inherited_idx)),
            "reset": int(len(reset_idx)),
            "lr": float(child_lrs[child_idx]),
            "dropout": float(child_dropouts[child_idx]),
        })

    # --------------------------
    # Fold ensemble evaluation
    # --------------------------

    with torch.no_grad():
        children_val_pred = predict_ensemble_mean_proba(children, Xn_val, Xc_val).detach().cpu().numpy()
        children_test_pred = predict_ensemble_mean_proba(children, Xn_test, Xc_test).detach().cpu().numpy()

    all_models_val_pred = (
        parent_val_pred + sum(
            predict_proba_model(m, Xn_val, Xc_val).detach().cpu().numpy()
            for m in children
        )
    ) / (1 + len(children))

    all_models_test_pred = (
        parent_test_pred + sum(
            predict_proba_model(m, Xn_test, Xc_test).detach().cpu().numpy()
            for m in children
        )
    ) / (1 + len(children))

    children_val_auc = auc_score(y_val_np, children_val_pred)
    all_models_val_auc = auc_score(y_val_np, all_models_val_pred)

    print("\n\n========================================================")
    print(f"================ FOLD {fold} SUMMARY ==================")
    print("========================================================")
    print(f"Parent VAL AUC        = {parent_val_auc:.6f}")
    print(f"Children mean VAL AUC = {children_val_auc:.6f}")
    print(f"All models VAL AUC    = {all_models_val_auc:.6f}")

    print("\nChild summary:")
    for h in child_val_scores:
        print(
            f"child={h['child']:02d} | "
            f"inherit={h['inherit']:03d} | reset={h['reset']:03d} | "
            f"lr={h['lr']:.6f} | drop={h['dropout']:.3f} | "
            f"solo_val={h['solo_val_auc']:.6f} | "
            f"parent_child_val={h['parent_child_val_auc']:.6f}"
        )

    fold_result = {
        "fold": fold,
        "val_idx": val_idx,
        "y_val": y_val_np.copy(),
        "parent_val_pred": parent_val_pred.copy(),
        "children_val_pred": children_val_pred.copy(),
        "all_models_val_pred": all_models_val_pred.copy(),
        "parent_test_pred": parent_test_pred.copy(),
        "children_test_pred": children_test_pred.copy(),
        "all_models_test_pred": all_models_test_pred.copy(),
        "parent_val_auc": float(parent_val_auc),
        "children_val_auc": float(children_val_auc),
        "all_models_val_auc": float(all_models_val_auc),
        "child_val_scores": child_val_scores,
    }

    del parent_model, children
    del Xn_train, Xc_train, y_train_t, Xn_val, Xc_val, y_val_t, Xn_test, Xc_test
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return fold_result

In [9]:
# ============================================================
# 6) RUN 5-FOLD CV
# ============================================================

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED,
)

fold_results = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_all_raw, y_all), start=1):
    fold_result = run_one_fold(fold, tr_idx, val_idx)
    fold_results.append(fold_result)



====================== FOLD 1/5 ======================


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(


fold train base: (351312, 40)
fold train final: (452683, 40)
fold val        : (87828, 40)
fold test       : (188165, 39)
cat_cols: 21
num_cols: 18
combo_names: ['Race_Compound_', 'Race_Year_']
Dnum = 18 | #cats = 21


/usr/local/lib/python3.12/dist-packages/rtdl_num_embeddings.py:340: UserWarning: The 1-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/rtdl_num_embeddings.py:340: UserWarning: The 17-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


USE_NUM_EMBEDDINGS fold: True
NUM_EMB_DIM_FLAT: 2304


===== FOLD 1 TRAIN PARENT =======
Parent stages = 7

------------------------------------------------
FOLD1_PARENT STAGE 1/7
------------------------------------------------
frozen=0 | trainable=300
[FOLD1_PARENT Stage 1] Epoch 01/4 | train_bce=0.325529 | val_AUC=0.939978 (best 0.939978@1)
[FOLD1_PARENT Stage 1] Epoch 02/4 | train_bce=0.249019 | val_AUC=0.946026 (best 0.946026@2)
[FOLD1_PARENT Stage 1] Epoch 03/4 | train_bce=0.233760 | val_AUC=0.947421 (best 0.947421@3)
[FOLD1_PARENT Stage 1] Epoch 04/4 | train_bce=0.222115 | val_AUC=0.948273 (best 0.948273@4)

------------------------------------------------
FOLD1_PARENT STAGE 2/7
------------------------------------------------
frozen=75 | trainable=225
[FOLD1_PARENT Stage 2] Epoch 01/4 | train_bce=0.214900 | val_AUC=0.948743 (best 0.948743@1)
[FOLD1_PARENT Stage 2] Epoch 02/4 | train_bce=0.204628 | val_AUC=0.948638 (best 0.948743@1)
[FOLD1_PARENT Stage 2] Epoch 03/4 | train_bce=

...

USE_NUM_EMBEDDINGS fold: True
NUM_EMB_DIM_FLAT: 2304


===== FOLD 4 TRAIN PARENT =======
Parent stages = 7

------------------------------------------------
FOLD4_PARENT STAGE 1/7
------------------------------------------------
frozen=0 | trainable=300
[FOLD4_PARENT Stage 1] Epoch 01/4 | train_bce=0.315488 | val_AUC=0.940168 (best 0.940168@1)
[FOLD4_PARENT Stage 1] Epoch 02/4 | train_bce=0.247851 | val_AUC=0.945712 (best 0.945712@2)
[FOLD4_PARENT Stage 1] Epoch 03/4 | train_bce=0.231747 | val_AUC=0.947446 (best 0.947446@3)
[FOLD4_PARENT Stage 1] Epoch 04/4 | train_bce=0.219706 | val_AUC=0.948452 (best 0.948452@4)

------------------------------------------------
FOLD4_PARENT STAGE 2/7
------------------------------------------------
frozen=75 | trainable=225
[FOLD4_PARENT Stage 2] Epoch 01/4 | train_bce=0.211537 | val_AUC=0.948308 (best 0.948308@1)
[FOLD4_PARENT Stage 2] Epoch 02/4 | train_bce=0.203561 | val_AUC=0.947871 (best 0.948308@1)
[FOLD4_PARENT Stage 2] Epoch 03/4 | train_bce=

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(


fold train base: (351312, 40)
fold train final: (452683, 40)
fold val        : (87828, 40)
fold test       : (188165, 39)
cat_cols: 21
num_cols: 18
combo_names: ['Race_Compound_', 'Race_Year_']
Dnum = 18 | #cats = 21


/usr/local/lib/python3.12/dist-packages/rtdl_num_embeddings.py:340: UserWarning: The 1-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/rtdl_num_embeddings.py:340: UserWarning: The 17-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


USE_NUM_EMBEDDINGS fold: True
NUM_EMB_DIM_FLAT: 2304


===== FOLD 5 TRAIN PARENT =======
Parent stages = 7

------------------------------------------------
FOLD5_PARENT STAGE 1/7
------------------------------------------------
frozen=0 | trainable=300
[FOLD5_PARENT Stage 1] Epoch 01/4 | train_bce=0.318927 | val_AUC=0.939528 (best 0.939528@1)
[FOLD5_PARENT Stage 1] Epoch 02/4 | train_bce=0.247542 | val_AUC=0.945829 (best 0.945829@2)
[FOLD5_PARENT Stage 1] Epoch 03/4 | train_bce=0.230109 | val_AUC=0.947446 (best 0.947446@3)
[FOLD5_PARENT Stage 1] Epoch 04/4 | train_bce=0.220178 | val_AUC=0.946889 (best 0.947446@3)

------------------------------------------------
FOLD5_PARENT STAGE 2/7
------------------------------------------------
frozen=75 | trainable=225
[FOLD5_PARENT Stage 2] Epoch 01/4 | train_bce=0.222849 | val_AUC=0.948273 (best 0.948273@1)
[FOLD5_PARENT Stage 2] Epoch 02/4 | train_bce=0.212127 | val_AUC=0.948267 (best 0.948273@1)
[FOLD5_PARENT Stage 2] Epoch 03/4 | train_bce=

In [10]:
# ============================================================
# 7) FINAL OOF + TEST AVERAGING
# ============================================================

oof_parent = np.zeros(len(train_raw), dtype=np.float64)
oof_children = np.zeros(len(train_raw), dtype=np.float64)
oof_all_models = np.zeros(len(train_raw), dtype=np.float64)

test_parent_sum = np.zeros(len(test_raw), dtype=np.float64)
test_children_sum = np.zeros(len(test_raw), dtype=np.float64)
test_all_models_sum = np.zeros(len(test_raw), dtype=np.float64)

for r in fold_results:
    val_idx = r["val_idx"]

    oof_parent[val_idx] = r["parent_val_pred"]
    oof_children[val_idx] = r["children_val_pred"]
    oof_all_models[val_idx] = r["all_models_val_pred"]

    test_parent_sum += r["parent_test_pred"] / N_FOLDS
    test_children_sum += r["children_test_pred"] / N_FOLDS
    test_all_models_sum += r["all_models_test_pred"] / N_FOLDS

oof_parent_auc = auc_score(y_all.values, oof_parent)
oof_children_auc = auc_score(y_all.values, oof_children)
oof_all_models_auc = auc_score(y_all.values, oof_all_models)

print("\n\n========================================================")
print("==================== FINAL CV SUMMARY =================")
print("========================================================")

print(f"N_FOLDS             = {N_FOLDS}")
print(f"N_CHILDREN_PER_FOLD = {N_CHILDREN_PER_FOLD}")
print(f"Total parents       = {N_FOLDS}")
print(f"Total children      = {N_FOLDS * N_CHILDREN_PER_FOLD}")
print(f"USE_NUM_EMBEDDINGS  = {USE_NUM_EMBEDDINGS}")

print("\nOOF AUC:")
print(f"  Parent mean      = {oof_parent_auc:.6f}")
print(f"  Children mean    = {oof_children_auc:.6f}")
print(f"  All models mean  = {oof_all_models_auc:.6f}")

print("\nFold AUCs:")
for r in fold_results:
    print(
        f"fold={r['fold']} | "
        f"parent={r['parent_val_auc']:.6f} | "
        f"children={r['children_val_auc']:.6f} | "
        f"all_models={r['all_models_val_auc']:.6f}"
    )



==================== FINAL CV SUMMARY =================
N_FOLDS             = 5
N_CHILDREN_PER_FOLD = 10
Total parents       = 5
Total children      = 50
USE_NUM_EMBEDDINGS  = True

OOF AUC:
  Parent mean      = 0.948100
  Children mean    = 0.951414
  All models mean  = 0.951374

Fold AUCs:
fold=1 | parent=0.948789 | children=0.952174 | all_models=0.952106
fold=2 | parent=0.948299 | children=0.951641 | all_models=0.951579
fold=3 | parent=0.947939 | children=0.950573 | all_models=0.950555
fold=4 | parent=0.948521 | children=0.951669 | all_models=0.951606
fold=5 | parent=0.948356 | children=0.951333 | all_models=0.951291


In [11]:
# ============================================================
# 8) CREATE SUBMISSIONS
# ============================================================

children_only_test_pred = np.clip(test_children_sum, 0.0, 1.0)
all_models_test_pred = np.clip(test_all_models_sum, 0.0, 1.0)
parent_only_test_pred = np.clip(test_parent_sum, 0.0, 1.0)

# Main submission: average of all fold children ensembles.
submission = pd.DataFrame({
    ID_COL: test_id.values,
    TARGET: children_only_test_pred,
})

submission.to_csv(OUTPUT_FILE, index=False)
submission.to_csv(OUTDIR / "submission_children_only.csv", index=False)

submission_all_models = pd.DataFrame({
    ID_COL: test_id.values,
    TARGET: all_models_test_pred,
})

submission_all_models.to_csv(OUTDIR / "submission_all_models_mean.csv", index=False)

submission_parent_only = pd.DataFrame({
    ID_COL: test_id.values,
    TARGET: parent_only_test_pred,
})

submission_parent_only.to_csv(OUTDIR / "submission_parent_only.csv", index=False)

print(f"\nSaved {OUTPUT_FILE}")
print("Saved submission_children_only.csv")
print("Saved submission_all_models_mean.csv")
print("Saved submission_parent_only.csv")

print("\nMain submission = average of all fold children ensembles")
print(submission.head())
print(submission[TARGET].describe())

print("\nAll-models diagnostic submission")
print(submission_all_models.head())
print(submission_all_models[TARGET].describe())

print("\nParent-only diagnostic submission")
print(submission_parent_only.head())
print(submission_parent_only[TARGET].describe())


Saved /kaggle/working/exp_20260514_028_parent_child_mlp_shared010_min/submission_exp_20260514_028_parent_child_mlp_shared010_min.csv
Saved submission_children_only.csv
Saved submission_all_models_mean.csv
Saved submission_parent_only.csv

Main submission = average of all fold children ensembles
       id  PitNextLap
0  439140    0.003342
1  439141    0.008766
2  439142    0.005178
3  439143    0.373695
4  439144    0.810393
count    188165.000000
mean          0.205562
std           0.315691
min           0.000282
25%           0.002090
50%           0.012781
75%           0.333835
max           0.992701
Name: PitNextLap, dtype: float64

All-models diagnostic submission
       id  PitNextLap
0  439140    0.003206
1  439141    0.008419
2  439142    0.005418
3  439143    0.365131
4  439144    0.810270
count    188165.000000
mean          0.205205
std           0.315317
min           0.000300
25%           0.002073
50%           0.012705
75%           0.332634
max           0.992415
Name

In [12]:
# ============================================================
# 9) SAVE OOF / PRED / CV RESULT / MEMO DRAFT
# ============================================================

def per_year_auc_from_raw(train_df, y_true, pred):
    out = {}
    years = sorted(pd.Series(train_df["Year"]).dropna().unique().tolist())
    y_arr = np.asarray(y_true).astype(int)
    p_arr = np.asarray(pred, dtype=float)
    for yr in years:
        mask = (train_df["Year"].values == yr)
        if mask.sum() == 0 or len(np.unique(y_arr[mask])) < 2:
            out[str(int(yr))] = None
        else:
            out[str(int(yr))] = float(roc_auc_score(y_arr[mask], p_arr[mask]))
    return out

children_per_year_auc = per_year_auc_from_raw(train_raw, y_all.values, oof_children)
parent_per_year_auc = per_year_auc_from_raw(train_raw, y_all.values, oof_parent)
all_models_per_year_auc = per_year_auc_from_raw(train_raw, y_all.values, oof_all_models)

# Main blend artifacts: children-only, following shared010 main submission.
np.save(OUTDIR / f"oof_{EXP_ID}.npy", oof_children.astype(np.float32))
np.save(OUTDIR / f"pred_{EXP_ID}.npy", children_only_test_pred.astype(np.float32))

# Diagnostics.
np.save(OUTDIR / f"oof_{EXP_ID}_parent_only.npy", oof_parent.astype(np.float32))
np.save(OUTDIR / f"pred_{EXP_ID}_parent_only.npy", parent_only_test_pred.astype(np.float32))
np.save(OUTDIR / f"oof_{EXP_ID}_all_models_mean.npy", oof_all_models.astype(np.float32))
np.save(OUTDIR / f"pred_{EXP_ID}_all_models_mean.npy", all_models_test_pred.astype(np.float32))

pd.DataFrame({ID_COL: train_ids_all.values, TARGET: oof_children}).to_csv(
    OUTDIR / f"oof_{EXP_ID}.csv", index=False
)

fold_scores_children = [float(r["children_val_auc"]) for r in fold_results]
fold_scores_parent = [float(r["parent_val_auc"]) for r in fold_results]
fold_scores_all = [float(r["all_models_val_auc"]) for r in fold_results]

cv_result = {
    "experiment": {
        "id": EXP_ID,
        "competition": "PS S6E5 Predicting F1 Pit Stops",
        "metric": "AUC",
        "created_at": "2026-05-14",
    },
    "source": {
        "reference_code": "code_shared_010.txt",
        "family": "ParentChildMLP",
        "description": "Parent MLP with progressive freezing + child models initialized from parent and partially reset.",
    },
    "config": {
        "seed": SEED,
        "n_folds": N_FOLDS,
        "n_children_per_fold": N_CHILDREN_PER_FOLD,
        "use_num_embeddings": bool(USE_NUM_EMBEDDINGS),
        "parent_hidden": PARENT_HIDDEN,
        "n_hidden_layers": N_HIDDEN_LAYERS,
        "batch_size": BATCH_SIZE,
        "parent_max_epochs_per_stage": PARENT_MAX_EPOCHS_PER_STAGE,
        "child_max_epochs_per_stage": CHILD_MAX_EPOCHS_PER_STAGE,
    },
    "result": {
        "parent_cv_auc": float(oof_parent_auc),
        "children_cv_auc": float(oof_children_auc),
        "all_models_cv_auc": float(oof_all_models_auc),
        "children_fold_scores": fold_scores_children,
        "parent_fold_scores": fold_scores_parent,
        "all_models_fold_scores": fold_scores_all,
        "children_fold_mean": float(np.mean(fold_scores_children)),
        "children_fold_std": float(np.std(fold_scores_children)),
        "children_per_year_auc": children_per_year_auc,
        "parent_per_year_auc": parent_per_year_auc,
        "all_models_per_year_auc": all_models_per_year_auc,
        "public_lb": None,
    },
    "artifacts": {
        "outdir": str(OUTDIR),
        "submission": str(OUTDIR / f"submission_{EXP_ID}.csv"),
        "oof_npy": str(OUTDIR / f"oof_{EXP_ID}.npy"),
        "pred_npy": str(OUTDIR / f"pred_{EXP_ID}.npy"),
        "cv_result": str(OUTDIR / "cv_result.json"),
    },
}

with open(OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(cv_result, f, ensure_ascii=False, indent=2)

memo_yml = f"""experiment:
  id: exp_20260514_028_parent_child_mlp_shared010_min
  title: Parent-Child MLP shared010 full
  competition: PS S6E5 Predicting F1 Pit Stops
  metric: AUC
  status: completed

objective:
  summary: >
    shared010 Parent-Child MLP ensembleをfullで再現し、
    既存RealMLP/TabM/GBDTとは異なるNN branchとしてblend寄与があるか確認する。
  intended_role: diversity_candidate

source:
  reference_code: code_shared_010.txt
  model_family: ParentChildMLP
  notes:
    - parent MLP with progressive neuron freezing
    - child models initialized from parent
    - random inherited neurons are kept and the remaining neurons are reset
    - rtdl_num_embeddings PiecewiseLinearEmbeddings are used when available

data:
  competition_train: /kaggle/input/competitions/playground-series-s6e5/train.csv
  competition_test: /kaggle/input/competitions/playground-series-s6e5/test.csv
  original: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv
  target: PitNextLap
  danger_col:
    name: Normalized_TyreLife
    used: false
    note: >
      Normalized_TyreLife is dropped if present.

model:
  family: ParentChildMLP
  seed: {SEED}
  folds: {N_FOLDS}
  children_per_fold: {N_CHILDREN_PER_FOLD}
  total_parents: {N_FOLDS}
  total_children: {N_FOLDS * N_CHILDREN_PER_FOLD}
  use_num_embeddings: {str(bool(USE_NUM_EMBEDDINGS)).lower()}
  params:
    parent_hidden: {PARENT_HIDDEN}
    n_hidden_layers: {N_HIDDEN_LAYERS}
    parent_dropout: {PARENT_DROPOUT}
    batch_size: {BATCH_SIZE}
    parent_lr: {PARENT_LR}
    parent_weight_decay: {PARENT_WEIGHT_DECAY}
    child_weight_decay: {CHILD_WEIGHT_DECAY}
    parent_max_epochs_per_stage: {PARENT_MAX_EPOCHS_PER_STAGE}
    child_max_epochs_per_stage: {CHILD_MAX_EPOCHS_PER_STAGE}

result:
  children_cv_auc: {float(oof_children_auc)}
  parent_cv_auc: {float(oof_parent_auc)}
  all_models_cv_auc: {float(oof_all_models_auc)}
  public_lb: null
  children_fold_scores: {[float(x) for x in fold_scores_children]}
  children_fold_mean: {float(np.mean(fold_scores_children))}
  children_fold_std: {float(np.std(fold_scores_children))}
  children_per_year_auc: {children_per_year_auc}

artifacts:
  notebook: exp_20260514_028_parent_child_mlp_shared010_min.ipynb
  oof: oof_exp_20260514_028_parent_child_mlp_shared010_min.npy
  pred: pred_exp_20260514_028_parent_child_mlp_shared010_min.npy
  submission: submission_exp_20260514_028_parent_child_mlp_shared010_min.csv
  cv_result: cv_result.json

blend_policy:
  benchmark_code: code_010_add_slsqp_drop020.txt
  add_to_stack:
    - "007"
    - "014"
    - "015"
    - "016"
    - "017"
    - "018"
    - "021"
    - "022"
    - "023"
    - "026"
    - "027"
    - "028"
  decision_focus:
    - single_cv_vs_021_027
    - corr_vs_021_tabm
    - corr_vs_realmlp_cluster
    - nm_hc_slsqp_weight
  keep_rule: >
    Keep if blend weight remains at least around 0.02,
    even if single CV is below the main RealMLP models.
  drop_rule: >
    Drop if correlation is high and blend weight is near zero.

judgement:
  status: pending
  summary: >
    CV/LB/correlation/blend weight確認後に keep / hold / drop を判断する。
"""
with open(OUTDIR / "memo_draft.yml", "w", encoding="utf-8") as f:
    f.write(memo_yml)

print("Saved OOF/PRED and memo_draft.yml to:", OUTDIR)

Saved OOF/PRED and memo_draft.yml to: /kaggle/working/exp_20260514_028_parent_child_mlp_shared010_min
